## Learning notes — 05. Create Customer Order Summary

### Learning objectives
Follow **05. Create Customer Order Summary** from its source inputs through its operations to the archived results.
- Explain join, ctas, aggregate, profiling as used in this lesson.

### Source-specific learning focus
- Relate inputs by keys and explain the resulting cardinality.
- Distinguish query results from materialized table state.
- Identify grouping keys and the business grain of summary measures.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.

SQL relation identifiers referenced in source (inputs and/or destinations; not a claim that these objects exist locally): `LIVE.silver_addresses`, `LIVE.silver_customers`, `LIVE.silver_orders`.


# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Create Customer Order Summary
1. Join the tables Silver_Customers, Silver_addresses and Silver_Orders
1. Retrieve the latest Address of the Customer 
1. Calculate the following values
    - total_orders
    - total_items_ordered
    - total_order_amount

##### CircuitBox Data Model
![dea06-05-create-customer-order-summary-2.png](images/dea06-05-create-customer-order-summary-2.png)

##### Customer Order Summary Requirement
![dea06-05-create-customer-order-summary-1.png](images/dea06-05-create-customer-order-summary-1.png)

## Learning notes — before source operation

**join**: Check join keys and cardinality on both inputs; unmatched keys and duplicate keys affect nulls and row counts in the joined output.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Two left rows and three right rows with the same matching key yield six joined rows.

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**aggregate**: Grouping changes the output grain to one row per group. Check null handling, counts versus distinct counts, and whether joins inflated measures before aggregation.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Grouping three orders for one customer by customer yields one customer-level group, not three order-level rows.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REFRESH MATERIALIZED VIEW gold_customer_order_summary
AS
SELECT c.customer_id,
       c.customer_name,
       c.date_of_birth,
       c.telephone,
       c.email,
       a.address_line_1,
       a.city,
       a.state,
       a.postcode,
       COUNT(DISTINCT o.order_id) AS total_orders,
       SUM(o.item_quantity) AS total_items_ordered,
       SUM(o.item_quantity * o.item_price) AS total_order_amount
  FROM LIVE.silver_customers c
  JOIN LIVE.silver_addresses a ON c.customer_id = a.customer_id
  JOIN LIVE.silver_orders o ON c.customer_id = o.customer_id
 WHERE a.__END_AT IS NULL
 GROUP BY ALL;
```